# Flutter v8 — yalnız "sonrası" ölçümü

`flutter-v8-qlora` üç adımı tek koşuda yapıyordu: temel model ölçümü, eğitim,
adapter ölçümü. İlk ikisi geçti, üçüncüsü peft'in torchao yoklaması yüzünden
düştü — adapter sağlam, eksik olan tek şey sayısı.

Bu notebook o son adımı, eğitimi tekrarlamadan koşuyor. Adapter önceki
koşunun çıktısından (`kernel_sources`) geliyor, script ve veri her zamanki
gibi dataset'ten.

Karşılaştırılacak temel — aynı 21 prompt, aynı greedy çözme:

```
Qwen/Qwen3-4B-Instruct-2507 (base)
  followed_unseen   71.4%  (n=7)
  followed_seen    100.0%  (n=6)
  clean             81.0%  (n=21)
  fenced           100.0%  (n=21)
  complete          95.2%  (n=21)
```

Temel model kanıt takibinde daha baştan yüksek çıktığı için `followed_*`
v8'i temelden ayırt edemiyor. Ayırt eden sayı **`clean`**: fine-tune'un asıl
işi ev stili, ve baseline 21 cevabın 4'ünde linter hatası veriyor.

In [ ]:
import os, glob, shutil, torch

cap = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "sm_%d%d" % cap)
assert cap >= (7, 5), (
    f"sm_{cap[0]}{cap[1]} yetersiz - T4 (sm_75) gerekiyor. "
    "Settings > Accelerator > GPU T4 x2")

In [ ]:
!pip -q install -U "transformers>=4.51" "peft>=0.11" "bitsandbytes>=0.43" "accelerate>=0.30" datasets 2>&1 | tail -2
import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)

# "Oncesi" sayilari 27 Tem 2026'da bu surumlerle uretildi. Farkli bir peft ya da
# transformers ile kosarsak iki sayi arasindaki farki adapter'a yazamayiz -
# kutuphane de degismis olur. Sessizce kaymaktansa burada yuksek sesle dursun.
TRAINED_WITH = {"transformers": "5.14.1", "peft": "0.19.1"}
now = {"transformers": transformers.__version__, "peft": peft.__version__}
assert now == TRAINED_WITH, (
    f"temel olcum {TRAINED_WITH} ile alinmisti, burada {now} var. "
    "Karsilastirmayi surdurmek icin surumleri sabitle, ya da temel olcumu "
    "bu surumlerle yeniden al.")

In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    print(root, "->", sorted(files)[:4], "..." if len(files) > 4 else "")
    if root.count("/") > 6:
        dirs.clear()

# Iki ayri girdi var ve ikisi de ayni dosya adlarini tasiyor: onceki kosu hem
# veriyi hem script'i /kaggle/working'e kopyalamisti, o dizin de simdi bu
# notebook'un girdisi. Yani ne "jsonl'i ara" ne "flutter_eval.py'yi ara"
# ayirt edici - ilki eski script'in yanindaki veriyi, ikincisi eski script'in
# kendisini secip olcumu sessizce kirik kodla kosardi.
#
# Ayirt edici olan tek sey mount yolunda gecen dataset slug'i. Derinlik yine
# sabitlenmiyor: mount bazen /kaggle/input/<slug>, bazen bir kat asagida.
hits = [p for p in glob.glob("/kaggle/input/**/flutter_eval.py", recursive=True)
        if "flutter-dataset" in p]
assert hits, "veri seti bagli degil - Add Input > emrahik/flutter-dataset"
SRC, WORK = os.path.dirname(hits[0]), "/kaggle/working"
assert glob.glob(f"{SRC}/flutter_screens_eval_v8*.jsonl"), (
    f"{SRC} script tasiyor ama olcum verisini tasimiyor")

# checkpoint-* dizinlerini ele: onlar ara adimlar, olculecek olan son adapter.
cands = [os.path.dirname(p) for p in
         glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)
         if "checkpoint-" not in p]
assert cands, ("adapter bagli degil - Add Input > Notebook Output > "
               "emrahik/flutter-v8-qlora")
ADAPTER = cands[0]
print("\nkaynak:", SRC, "\nadapter:", ADAPTER)

os.makedirs(f"{WORK}/data", exist_ok=True)
for f in os.listdir(SRC):
    dst = f"{WORK}/data/{f}" if f.endswith(".jsonl") else f"{WORK}/{f}"
    shutil.copy(f"{SRC}/{f}", dst)
os.chdir(WORK)

# Kosan script gercekten dataset'ten gelen duzeltilmis surum mu: torchao
# kalkani yoksa olcum yine adapter yuklerken duser.
assert "_silence_torchao_probe" in open(f"{WORK}/flutter_eval.py").read(), (
    "flutter_eval.py torchao kalkanini tasimiyor - dataset surumu eski")

# Girdi salt-okunur; peft adapter dizinine yazmaya calisirsa dusmesin.
shutil.copytree(ADAPTER, f"{WORK}/adapter", dirs_exist_ok=True)
print(sorted(os.listdir(f"{WORK}/adapter")))

## Sonrası — adapter ile

Temel koşuyla aynı promptlar, aynı `--max-new-tokens 1200`, aynı greedy
çözme. Tek fark model.

In [ ]:
!python flutter_eval.py --backend hf \
    --base-model Qwen/Qwen3-4B-Instruct-2507 \
    --adapter adapter \
    --eval data/flutter_screens_eval_v8.jsonl \
    --meta data/flutter_screens_eval_v8_meta.jsonl \
    --max-new-tokens 1200 \
    --dump after.jsonl

`after.jsonl` ham tamamlamaları taşıyor. Bir sayı şüpheli görünürse modelin
ne yazdığına bakılacak yer orası — `before.jsonl` de önceki koşunun
çıktısında duruyor, ikisi satır satır aynı sırada.